# GLiNER2: OntoNotes5 + `combined_output.jsonl` preprocessing and fine-tuning

This notebook builds a unified 7-label NER model:

- `PERSON`
- `ORG`
- `GPE`
- `EVENT`
- `DATE`
- `TIME`
- `QUANTITY`

**Source authority**

| Label | Trusted source |
|---|---|
| PERSON | TNER OntoNotes5 |
| ORG | TNER OntoNotes5 |
| DATE | TNER OntoNotes5 |
| TIME | TNER OntoNotes5 |
| QUANTITY | TNER OntoNotes5 |
| GPE | `combined_output.jsonl` |
| EVENT | `combined_output.jsonl` |

The notebook deliberately does **not** treat labels omitted from a source as false entities. It first creates source-specific datasets, then optionally trains two specialist GLiNER2 adapters and uses them to pseudo-label the missing label families before final unified training.

GLiNER2's documented training API uses `InputExample(text=..., entities={label: [span_texts...]})` and supports LoRA training. citeturn0search1turn1search1

In [ ]:
# 0. Install dependencies
# Run this cell in Colab.

!pip -q install -U "gliner2[local]" datasets scikit-learn pandas matplotlib tqdm


In [ ]:
# 1. Configuration

from pathlib import Path
import json
import random
import re
from collections import Counter, defaultdict

SEED = 42
random.seed(SEED)

# Upload combined_output.jsonl to Colab or change this path.
DATASET2_PATH = Path("/content/combined_output.jsonl")

# Output directory
WORK_DIR = Path("/content/gliner2_ner")
WORK_DIR.mkdir(parents=True, exist_ok=True)

# Final ontology
FINAL_LABELS = [
    "PERSON", "ORG", "GPE", "EVENT", "DATE", "TIME", "QUANTITY"
]

# OntoNotes5 labels we trust
ONTONOTES_KEEP = {
    "PERSON", "ORG", "DATE", "TIME", "QUANTITY"
}

# Dataset 2 labels we trust
DATASET2_KEEP = {
    "GPE", "EVENT"
}

# Optional limits. Set to None to use everything.
MAX_ONTONOTES = None
MAX_DATASET2 = None

# Sampling caps for final training.
# None = no cap. These are conservative defaults; inspect counts first.
MAX_PER_LABEL = None

# Specialist pseudo-labeling:
# Set False if you want a strictly supervised-only final dataset.
USE_PSEUDO_LABELING = True
PSEUDO_THRESHOLD = 0.85

print("Work dir:", WORK_DIR)
print("Final labels:", FINAL_LABELS)


## 2. Load OntoNotes5 and inspect its tag mapping

The TNER OntoNotes5 records contain `tokens` plus numeric `tags`. The notebook reads the `ClassLabel` metadata from the Hugging Face dataset instead of hard-coding numeric tag IDs. This is important because the numeric IDs should never be guessed.

In [ ]:
from datasets import load_dataset

ontonotes = load_dataset("tner/ontonotes5")

print(ontonotes)
print(ontonotes["train"].features)

# Find the tag feature and print its label names.
tag_feature = ontonotes["train"].features["tags"]
print("\nTag feature:", tag_feature)

if hasattr(tag_feature, "feature") and hasattr(tag_feature.feature, "names"):
    ID2TAG = dict(enumerate(tag_feature.feature.names))
elif hasattr(tag_feature, "names"):
    ID2TAG = dict(enumerate(tag_feature.names))
else:
    raise RuntimeError("Could not find OntoNotes tag names in the dataset features.")

print("\nFirst tag IDs:")
for k, v in list(ID2TAG.items())[:40]:
    print(k, "->", v)


In [ ]:
# 3. Inspect label distribution in OntoNotes5

def normalize_ontonotes_label(tag_name):
    # Accept B-PERSON / I-PERSON, B-PER / I-PER, etc.
    if "-" in tag_name:
        prefix, label = tag_name.split("-", 1)
    else:
        prefix, label = "O", tag_name

    aliases = {
        "PER": "PERSON",
        "PERSON": "PERSON",
        "ORG": "ORG",
        "DATE": "DATE",
        "TIME": "TIME",
        "QUANTITY": "QUANTITY",
    }
    return aliases.get(label, label)

onto_counts = Counter()

for split_name, split in ontonotes.items():
    for row in split:
        for tag_id in row["tags"]:
            tag_name = ID2TAG[int(tag_id)]
            if tag_name == "O":
                continue
            onto_counts[normalize_ontonotes_label(tag_name)] += 1

print("OntoNotes entity-token counts:")
for label, count in onto_counts.most_common():
    print(f"{label:15s} {count:,}")


## 4. Convert OntoNotes5 BIO tags to entity spans

Only `PERSON`, `ORG`, `DATE`, `TIME`, and `QUANTITY` are retained.

All other OntoNotes entity types are ignored because they are not authoritative for this project.

In [ ]:
def bio_to_entities(tokens, tag_ids, id2tag):
    """Convert BIO/BIOES-like token tags into (start, end_exclusive, label)."""
    entities = []
    current = None

    def close_current(end_idx):
        nonlocal current
        if current is not None:
            start, label = current
            entities.append((start, end_idx, label))
            current = None

    for i, tag_id in enumerate(tag_ids):
        tag = id2tag[int(tag_id)]

        if tag == "O":
            close_current(i)
            continue

        if "-" in tag:
            prefix, raw_label = tag.split("-", 1)
        else:
            prefix, raw_label = "B", tag

        label = normalize_ontonotes_label(raw_label)

        # Ignore labels outside our trusted OntoNotes ontology.
        if label not in ONTONOTES_KEEP:
            close_current(i)
            continue

        if prefix in {"B", "S"}:
            close_current(i)
            if prefix == "S":
                entities.append((i, i + 1, label))
            else:
                current = (i, label)

        elif prefix in {"I", "M"}:
            if current is None or current[1] != label:
                close_current(i)
                current = (i, label)

        elif prefix == "E":
            if current is None or current[1] != label:
                close_current(i)
                entities.append((i, i + 1, label))
            else:
                close_current(i + 1)

        else:
            close_current(i)

    close_current(len(tokens))
    return entities


def entity_text(tokens, start, end):
    return " ".join(tokens[start:end]).strip()


def convert_ontonotes_row(row):
    tokens = row["tokens"]
    spans = bio_to_entities(tokens, row["tags"], ID2TAG)

    entities = []
    for start, end, label in spans:
        text = entity_text(tokens, start, end)
        if text:
            entities.append([start, end - 1, label])

    return {
        "tokenized_text": tokens,
        "ner": entities,
        "source": "ontonotes5"
    }


# Test on the first training row
sample = convert_ontonotes_row(ontonotes["train"][0])
print(json.dumps(sample, indent=2))


In [ ]:
# 5. Load and normalize Dataset 2

if not DATASET2_PATH.exists():
    raise FileNotFoundError(
        f"{DATASET2_PATH} not found. Upload combined_output.jsonl to Colab "
        "or change DATASET2_PATH in the configuration cell."
    )

dataset2 = []
with DATASET2_PATH.open("r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
        except json.JSONDecodeError as e:
            print(f"Skipping malformed JSONL line {line_no}: {e}")
            continue

        tokens = obj.get("tokenized_text")
        ner = obj.get("ner", [])

        if not isinstance(tokens, list) or not isinstance(ner, list):
            continue

        # Keep ONLY GPE/EVENT. Do not keep Dataset 2 PERSON/ORG.
        filtered = []
        for ann in ner:
            if len(ann) != 3:
                continue
            start, end, label = ann
            label = str(label).upper()
            if label not in DATASET2_KEEP:
                continue

            start, end = int(start), int(end)
            if start < 0 or end < start or end >= len(tokens):
                continue

            filtered.append([start, end, label])

        dataset2.append({
            "tokenized_text": tokens,
            "ner": filtered,
            "source": "dataset2"
        })

print("Dataset 2 records:", len(dataset2))
print("Dataset 2 label counts:", Counter(a[2] for r in dataset2 for a in r["ner"]))

print("\nExample:")
print(json.dumps(dataset2[0], indent=2))


In [ ]:
# 6. Build cleaned source datasets

def has_trusted_entity(record):
    return bool(record["ner"])


onto_records = []
for split_name, split in ontonotes.items():
    for row in split:
        rec = convert_ontonotes_row(row)
        if has_trusted_entity(rec):
            rec["source_split"] = split_name
            onto_records.append(rec)

dataset2_records = [r for r in dataset2 if has_trusted_entity(r)]

if MAX_ONTONOTES is not None:
    random.shuffle(onto_records)
    onto_records = onto_records[:MAX_ONTONOTES]

if MAX_DATASET2 is not None:
    random.shuffle(dataset2_records)
    dataset2_records = dataset2_records[:MAX_DATASET2]

print("OntoNotes usable records:", len(onto_records))
print("Dataset 2 usable records:", len(dataset2_records))

def count_entities(records):
    c = Counter()
    for r in records:
        for _, _, label in r["ner"]:
            c[label] += 1
    return c

print("\nOntoNotes:")
print(count_entities(onto_records))

print("\nDataset 2:")
print(count_entities(dataset2_records))


## 7. Important: preserve mixed sentences

We **do not throw away** a Dataset 2 sentence just because it contains PERSON/ORG.

For example, if the source contains:

`PERSON + ORG + GPE + EVENT`

we retain the full sentence and the trusted `GPE/EVENT` annotations.

The notebook does **not** claim the omitted PERSON/ORG are non-entities. They are simply outside that source's trusted annotation scope.

Because the public GLiNER2 training API documents ordinary entity supervision rather than a per-example label-mask parameter, the notebook uses a conservative two-stage strategy below:

1. train a PERSON/ORG/DATE/TIME/QUANTITY specialist on OntoNotes;
2. train a GPE/EVENT specialist on Dataset 2;
3. optionally pseudo-label the missing label families on the opposite source using high-confidence predictions;
4. train one final unified GLiNER2 model on the enriched data.

This avoids discarding mixed sentences while reducing the false-negative problem from simply deleting annotations. GLiNER2's documented training interface accepts entity dictionaries and supports LoRA adapters. citeturn0search1turn1search1

In [ ]:
# 8. Convert token-span records into GLiNER2 InputExample objects

from gliner2.training.data import InputExample

def record_to_input_example(record):
    tokens = record["tokenized_text"]
    text = " ".join(tokens)

    grouped = defaultdict(list)
    for start, end, label in record["ner"]:
        span = " ".join(tokens[start:end + 1]).strip()
        if span:
            grouped[label].append(span)

    # Remove duplicate strings within a label.
    entities = {
        label: list(dict.fromkeys(spans))
        for label, spans in grouped.items()
        if spans
    }

    return InputExample(text=text, entities=entities)


onto_examples = [record_to_input_example(r) for r in onto_records]
dataset2_examples = [record_to_input_example(r) for r in dataset2_records]

print(onto_examples[0])
print(dataset2_examples[0])


In [ ]:
# 9. Save cleaned source JSONL in GLiNER2's documented JSONL shape

def examples_to_jsonl(examples, path):
    with open(path, "w", encoding="utf-8") as f:
        for ex in examples:
            obj = {
                "input": ex.text,
                "output": {
                    "entities": ex.entities
                }
            }
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")


ONTO_JSONL = WORK_DIR / "ontonotes_trusted.jsonl"
D2_JSONL = WORK_DIR / "dataset2_trusted.jsonl"

examples_to_jsonl(onto_examples, ONTO_JSONL)
examples_to_jsonl(dataset2_examples, D2_JSONL)

print(ONTO_JSONL)
print(D2_JSONL)


In [ ]:
# 10. Label statistics before balancing

import pandas as pd

source_counts = pd.DataFrame([
    {"source": "OntoNotes5", "label": label, "count": count}
    for label, count in count_entities(onto_records).items()
] + [
    {"source": "Dataset2", "label": label, "count": count}
    for label, count in count_entities(dataset2_records).items()
])

display(source_counts.sort_values(["label", "source"]))

total_counts = Counter()
for c in (count_entities(onto_records), count_entities(dataset2_records)):
    total_counts.update(c)

print("\nTotal trusted entity counts:")
for label in FINAL_LABELS:
    print(f"{label:12s}: {total_counts[label]:,}")


## 11. Optional label-aware cap

Do not blindly undersample to the smallest class. The function below only applies a cap if you set `MAX_PER_LABEL`.

It keeps examples containing rare labels and limits excessive duplication of dominant labels. The default is `None`, so no cap is applied until you inspect the statistics.

In [ ]:
def cap_records_by_label(records, max_per_label=None, seed=42):
    if max_per_label is None:
        return list(records)

    rng = random.Random(seed)

    by_label = defaultdict(list)
    for rec in records:
        labels = set(a[2] for a in rec["ner"])
        for label in labels:
            by_label[label].append(rec)

    selected_ids = set()

    # First ensure each label can contribute up to the cap.
    for label in FINAL_LABELS:
        candidates = list(by_label.get(label, []))
        rng.shuffle(candidates)
        for rec in candidates[:max_per_label]:
            selected_ids.add(id(rec))

    return [r for r in records if id(r) in selected_ids]


onto_balanced = cap_records_by_label(onto_records, MAX_PER_LABEL, SEED)
d2_balanced = cap_records_by_label(dataset2_records, MAX_PER_LABEL, SEED)

print(len(onto_balanced), len(d2_balanced))


# 12. Train the two specialist GLiNER2 LoRA adapters

This is the key quality-control step.

- **OntoNotes adapter:** learns `PERSON`, `ORG`, `DATE`, `TIME`, `QUANTITY`.
- **Dataset 2 adapter:** learns `GPE`, `EVENT`.

LoRA is used to keep training lighter; GLiNER2 documents LoRA as parameter-efficient fine-tuning and reports substantially fewer trainable parameters. citeturn1search1

In [ ]:
from gliner2 import GLiNER2
from gliner2.training.trainer import GLiNER2Trainer, TrainingConfig

BASE_MODEL = "fastino/gliner2-base-v1"

def make_config(output_dir, experiment_name, epochs=2, batch_size=8):
    import torch

    return TrainingConfig(
        output_dir=str(output_dir),
        experiment_name=experiment_name,
        num_epochs=epochs,
        batch_size=batch_size,
        gradient_accumulation_steps=2,
        encoder_lr=1e-5,
        task_lr=5e-4,
        warmup_ratio=0.1,
        scheduler_type="cosine",
        eval_strategy="no",
        logging_steps=50,

        # LoRA
        use_lora=True,
        lora_r=8,
        lora_alpha=16.0,
        lora_dropout=0.0,
        lora_target_modules=["encoder"],
        save_adapter_only=True,

        # Use fp16 only when CUDA is available.
        fp16=torch.cuda.is_available(),
    )


ONTO_ADAPTER_DIR = WORK_DIR / "adapters" / "ontonotes"
D2_ADAPTER_DIR = WORK_DIR / "adapters" / "dataset2"

print("CUDA available:", __import__("torch").cuda.is_available())


In [ ]:
# 13. Train OntoNotes specialist
# Start with 1-2 epochs to verify the pipeline.
# Increase after the data pipeline is validated.

onto_train_examples = [record_to_input_example(r) for r in onto_balanced]

onto_model = GLiNER2.from_pretrained(BASE_MODEL)
onto_config = make_config(ONTO_ADAPTER_DIR, "ontonotes_trusted", epochs=2, batch_size=8)
onto_trainer = GLiNER2Trainer(model=onto_model, config=onto_config)

onto_train_result = onto_trainer.train(train_data=onto_train_examples)
print(onto_train_result)
print("Expected adapter:", ONTO_ADAPTER_DIR / "final")


In [ ]:
# 14. Train Dataset 2 specialist

d2_train_examples = [record_to_input_example(r) for r in d2_balanced]

d2_model = GLiNER2.from_pretrained(BASE_MODEL)
d2_config = make_config(D2_ADAPTER_DIR, "dataset2_gpe_event", epochs=2, batch_size=8)
d2_trainer = GLiNER2Trainer(model=d2_model, config=d2_config)

d2_train_result = d2_trainer.train(train_data=d2_train_examples)
print(d2_train_result)
print("Expected adapter:", D2_ADAPTER_DIR / "final")


# 15. Optional pseudo-labeling to recover mixed sentences

This stage uses each specialist to fill the label families it did **not** learn from that source:

- Dataset 1 → pseudo-label `GPE` + `EVENT`
- Dataset 2 → pseudo-label `PERSON` + `ORG` + `DATE` + `TIME` + `QUANTITY`

Only predictions above `PSEUDO_THRESHOLD` are accepted.

This is still **weak supervision**, so inspect a sample before final training. If the specialist predictions are poor, set `USE_PSEUDO_LABELING = False` and train the final model only on trusted labels.

In [ ]:
# 16. Pseudo-label helper

def pseudo_label_records(model, records, labels, threshold=0.85, batch_size=8):
    """Add high-confidence predictions to tokenized records.

    GLiNER2 returns character spans. We map those spans back to the
    whitespace-tokenized representation used by our records.
    """
    texts = [" ".join(r["tokenized_text"]) for r in records]

    results = model.batch_extract_entities(
        texts,
        labels,
        include_confidence=True,
        include_spans=True,
        batch_size=batch_size
    )

    enriched = []

    for rec, result in zip(records, results):
        new_rec = {
            "tokenized_text": list(rec["tokenized_text"]),
            "ner": [list(x) for x in rec["ner"]],
            "source": rec.get("source", ""),
            "source_split": rec.get("source_split")
        }

        text = " ".join(new_rec["tokenized_text"])

        # Build character offsets for each token.
        token_char_spans = []
        cursor = 0
        for tok in new_rec["tokenized_text"]:
            start = cursor
            end = start + len(tok)
            token_char_spans.append((start, end))
            cursor = end + 1

        for label, predictions in result.get("entities", {}).items():
            for pred in predictions:
                if isinstance(pred, dict):
                    conf = float(pred.get("confidence", 0.0))
                    pstart = pred.get("start")
                    pend = pred.get("end")
                    ptext = pred.get("text", "")
                else:
                    # Defensive fallback for alternate API shapes.
                    continue

                if conf < threshold or pstart is None or pend is None:
                    continue

                # Map character span to token span.
                overlapping = []
                for i, (ts, te) in enumerate(token_char_spans):
                    if te > pstart and ts < pend:
                        overlapping.append(i)

                if not overlapping:
                    continue

                start_tok = overlapping[0]
                end_tok = overlapping[-1]

                new_rec["ner"].append([start_tok, end_tok, label])

        # Deduplicate annotations.
        seen = set()
        clean = []
        for ann in new_rec["ner"]:
            key = tuple(ann)
            if key not in seen:
                seen.add(key)
                clean.append(ann)

        new_rec["ner"] = clean
        enriched.append(new_rec)

    return enriched


if USE_PSEUDO_LABELING:
    # Load trained specialists from their final adapter directories.
    onto_teacher = GLiNER2.from_pretrained(BASE_MODEL)
    onto_teacher.load_adapter(str(ONTO_ADAPTER_DIR / "final"))

    d2_teacher = GLiNER2.from_pretrained(BASE_MODEL)
    d2_teacher.load_adapter(str(D2_ADAPTER_DIR / "final"))

    # Dataset 1 gets GPE/EVENT predictions from Dataset 2 specialist.
    onto_enriched = pseudo_label_records(
        d2_teacher,
        onto_records,
        labels=["GPE", "EVENT"],
        threshold=PSEUDO_THRESHOLD,
        batch_size=8
    )

    # Dataset 2 gets PERSON/ORG/DATE/TIME/QUANTITY predictions
    # from OntoNotes specialist.
    d2_enriched = pseudo_label_records(
        onto_teacher,
        dataset2_records,
        labels=["PERSON", "ORG", "DATE", "TIME", "QUANTITY"],
        threshold=PSEUDO_THRESHOLD,
        batch_size=8
    )

    print("Pseudo-labeling complete.")
else:
    onto_enriched = onto_records
    d2_enriched = dataset2_records
    print("Pseudo-labeling disabled.")


In [ ]:
# 17. Inspect pseudo-labeling before final training

def show_examples(records, n=5):
    for i, rec in enumerate(records[:n]):
        print("=" * 80)
        print(" ".join(rec["tokenized_text"]))
        for start, end, label in rec["ner"]:
            span = " ".join(rec["tokenized_text"][start:end+1])
            print(f"  {label:10s}: {span}")

print("Dataset 1 enriched examples:")
show_examples(onto_enriched, 5)

print("\nDataset 2 enriched examples:")
show_examples(d2_enriched, 5)


In [ ]:
# 18. Final combined dataset + global deduplication

def record_key(rec):
    return " ".join(rec["tokenized_text"]).strip().lower()

combined_records = []
seen = set()

for rec in onto_enriched + d2_enriched:
    key = record_key(rec)
    if not key:
        continue

    # Merge duplicate text records instead of keeping conflicting duplicates.
    if key not in seen:
        seen.add(key)
        combined_records.append(rec)
    else:
        # Merge annotations into the existing record.
        existing = next(r for r in combined_records if record_key(r) == key)
        existing["ner"].extend(rec["ner"])
        existing["ner"] = [list(x) for x in {
            tuple(a) for a in existing["ner"]
        }]

print("Final records:", len(combined_records))
print("Final entity counts:", count_entities(combined_records))


# 19. Split train / validation / test

The split is performed **after** source normalization and deduplication.

This is important because otherwise the same or nearly identical text can appear in different splits.

In [ ]:
from sklearn.model_selection import train_test_split

random.Random(SEED).shuffle(combined_records)

train_records, temp_records = train_test_split(
    combined_records,
    test_size=0.20,
    random_state=SEED
)

val_records, test_records = train_test_split(
    temp_records,
    test_size=0.50,
    random_state=SEED
)

print("train:", len(train_records))
print("val:  ", len(val_records))
print("test: ", len(test_records))

print("\nTrain counts:")
print(count_entities(train_records))
print("\nValidation counts:")
print(count_entities(val_records))
print("\nTest counts:")
print(count_entities(test_records))


In [ ]:
# 20. Save final GLiNER2 JSONL files

TRAIN_JSONL = WORK_DIR / "train.jsonl"
VAL_JSONL = WORK_DIR / "validation.jsonl"
TEST_JSONL = WORK_DIR / "test.jsonl"

examples_to_jsonl([record_to_input_example(r) for r in train_records], TRAIN_JSONL)
examples_to_jsonl([record_to_input_example(r) for r in val_records], VAL_JSONL)
examples_to_jsonl([record_to_input_example(r) for r in test_records], TEST_JSONL)

print("Saved:")
print(TRAIN_JSONL)
print(VAL_JSONL)
print(TEST_JSONL)


# 21. Validate the GLiNER2 training dataset

GLiNER2 provides `TrainingDataset` validation utilities. We use them before spending GPU time on final training. citeturn0search4

In [ ]:
from gliner2.training.data import TrainingDataset

train_examples = [record_to_input_example(r) for r in train_records]
val_examples = [record_to_input_example(r) for r in val_records]

train_dataset = TrainingDataset(train_examples)
val_dataset = TrainingDataset(val_examples)

train_dataset.validate(strict=True, raise_on_error=True)
val_dataset.validate(strict=True, raise_on_error=True)

print("Training dataset:")
train_dataset.print_stats()

print("\nValidation dataset:")
val_dataset.print_stats()


# 22. Final unified GLiNER2 fine-tuning

Start with a short run to verify the pipeline. For a serious run, increase `num_epochs` after inspecting validation performance.

The current GLiNER2 documentation uses `GLiNER2.from_pretrained("fastino/gliner2-base-v1")`, `GLiNER2Trainer`, and `TrainingConfig`; LoRA is supported through the same training configuration. citeturn0search1turn1search1

In [ ]:
FINAL_MODEL_DIR = WORK_DIR / "final_model"

# Recommended starting configuration for Colab.
FINAL_EPOCHS = 3
FINAL_BATCH_SIZE = 8

final_model = GLiNER2.from_pretrained(BASE_MODEL)

final_config = TrainingConfig(
    output_dir=str(FINAL_MODEL_DIR),
    experiment_name="news_7label_ner",
    num_epochs=FINAL_EPOCHS,
    batch_size=FINAL_BATCH_SIZE,
    gradient_accumulation_steps=2,
    encoder_lr=1e-5,
    task_lr=5e-4,
    warmup_ratio=0.1,
    scheduler_type="cosine",
    eval_strategy="epoch",
    save_best=True,
    early_stopping=True,
    early_stopping_patience=2,
    logging_steps=50,

    # LoRA
    use_lora=True,
    lora_r=8,
    lora_alpha=16.0,
    lora_dropout=0.0,
    lora_target_modules=["encoder"],
    save_adapter_only=True,

    # Mixed precision on GPU.
    fp16=__import__("torch").cuda.is_available(),
)

final_trainer = GLiNER2Trainer(
    model=final_model,
    config=final_config
)

final_result = final_trainer.train(
    train_data=train_examples,
    val_data=val_examples
)

print(final_result)
print("Final output:", FINAL_MODEL_DIR)


# 23. Load the trained model and test it

Use the seven canonical labels at inference time. GLiNER2 supports entity extraction with explicit entity types and optional confidence/spans. citeturn2search0

In [ ]:
# Load the final model/adapter.
# If save_adapter_only=True, the adapter is under FINAL_MODEL_DIR / "final".
# Otherwise load the best/full model directory.

INFERENCE_MODEL_PATH = FINAL_MODEL_DIR / "final"

inference_model = GLiNER2.from_pretrained(BASE_MODEL)

try:
    inference_model.load_adapter(str(INFERENCE_MODEL_PATH))
    print("Loaded LoRA adapter:", INFERENCE_MODEL_PATH)
except Exception as e:
    print("Adapter load failed; trying to load full model directory.")
    print("Reason:", e)
    inference_model = GLiNER2.from_pretrained(str(INFERENCE_MODEL_PATH))


ENTITY_TYPES = {
    "PERSON": "Names of individual people.",
    "ORG": "Organizations, companies, institutions, agencies, or groups.",
    "GPE": "Geopolitical entities such as countries, states, provinces, and cities.",
    "EVENT": "Named or identifiable real-world events, conflicts, operations, disasters, or incidents.",
    "DATE": "Calendar dates and date expressions.",
    "TIME": "Clock times and time-of-day expressions.",
    "QUANTITY": "Measured quantities, amounts, counts, or numeric quantities with units."
}

test_text = (
    "President Donald Trump met officials from NATO in Washington on "
    "August 14, 2026 at 5 PM during the Iran conflict."
)

result = inference_model.extract_entities(
    test_text,
    ENTITY_TYPES,
    include_confidence=True,
    include_spans=True
)

print(json.dumps(result, indent=2, ensure_ascii=False))


# 24. Simple exact-match evaluation

This evaluation is intentionally simple: normalized `(entity text, label)` exact matching.

For production evaluation, use a manually verified test set whose annotation policy is consistent across all seven labels.

In [ ]:
def normalize_text(s):
    return re.sub(r"\s+", " ", s.strip().lower())

def evaluate_record(model, record):
    text = " ".join(record["tokenized_text"])

    gold = set()
    for start, end, label in record["ner"]:
        span = " ".join(record["tokenized_text"][start:end+1])
        gold.add((normalize_text(span), label))

    pred_result = model.extract_entities(
        text,
        ENTITY_TYPES,
        include_confidence=False,
        include_spans=False
    )

    pred = set()
    for label, values in pred_result.get("entities", {}).items():
        label = label.upper()
        for value in values:
            if isinstance(value, dict):
                value = value.get("text", "")
            pred.add((normalize_text(str(value)), label))

    tp = len(gold & pred)
    fp = len(pred - gold)
    fn = len(gold - pred)

    return tp, fp, fn


def evaluate_model(model, records, max_examples=500):
    records = records[:max_examples]

    tp = fp = fn = 0
    for rec in records:
        a, b, c = evaluate_record(model, rec)
        tp += a
        fp += b
        fn += c

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "examples": len(records)
    }


metrics = evaluate_model(inference_model, test_records, max_examples=500)
metrics


## 25. Recommended training progression

For your actual Colab run:

1. Run preprocessing and inspect the label counts.
2. Train the two specialist adapters for 1–2 epochs.
3. Inspect 50–100 pseudo-labeled examples.
4. If pseudo-label quality is poor, lower nothing blindly — instead set `USE_PSEUDO_LABELING=False` and use trusted-only supervision.
5. Run the final unified model for 3 epochs first.
6. Evaluate each label separately on a manually checked test set.
7. Only then increase epochs/data.

**Important:** `TIME`, `EVENT`, and `QUANTITY` should be evaluated independently because aggregate F1 can hide poor performance on minority labels.

GLiNER2's current PyPI release is 1.3.2, and the official training documentation supports local training plus LoRA. citeturn0search0turn1search1